# QC/trimming samples 
- (sequenced Jan 2024)
- PSTR, OFAV, OANN, MCAV, MMEA
- 2019 and 2022 

In [ ]:
# unzip files, make sample list, rename files in dir

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=50G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 24:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs012024/unziplist%j.out  # %j = job ID

# unzip files
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/raw

# unzip 
gunzip *.fastq.gz

## Get sample list 
ls *.fastq* | sed -E 's/_S[0-9]+_R[12]_001\.fastq.*//' | sort -u > /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/sampleids.txt

# remove sequencer ID from actual file names 
for file in *; do
    new_name=$(echo "$file" | sed 's/_S[0-9]\+//')
    mv "$file" "$new_name"
done

# job id- 51476024

In [ ]:
# qc

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=50G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 24:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs012024/slurm-qc-%j.out  # %j = job ID

module load conda/latest
conda activate qc

# Define the paths and variables
FILEPATH='/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/raw'
OUTPUT_RESULTS='/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed' 
NSLOTS=4  
SAMPLE_NAMES_FILE="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/sampleids.txt"

cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/raw
# Check if the file exists
if [ ! -e "$SAMPLE_NAMES_FILE" ]; then
    echo "Error: $SAMPLE_NAMES_FILE does not exist."
    exit 1
fi

# Read each line from the file and perform actions
while IFS= read -r sample_id; do
    # Form the full file names
    input_r1="$FILEPATH/${sample_id}_R1_001.fastq"
    input_r2="$FILEPATH/${sample_id}_R2_001.fastq"
    
    # Ensure the input files exist before running the tools
    if [ ! -e "$input_r1" ] || [ ! -e "$input_r2" ]; then
        echo "Error: Input files do not exist for sample $sample_id"
        continue
    fi

    # Run trim_galore
    trim_galore -j "$NSLOTS" -q 20 --phred33 --length 20 --paired $input_r1 $input_r2 --fastqc -o $OUTPUT_RESULTS --dont_gzip

done < "$SAMPLE_NAMES_FILE"

# run multiqc (summary of fastqc) 
conda deactivate
conda activate multiqc
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed
multiqc .
    
# calculate read depth of raw and trimmed files 
# create reads count file 
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw
echo "sample,read_count,step">reads.csv

# raw reads 
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/raw
for FILE in *.fastq; do
    NAME=$(basename "$FILE")
    COUNT=$(( $(wc -l < "$FILE") / 4 ))
    echo "$NAME,$COUNT,raw" >> ../reads.csv
done

# trimmed reads 
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed
for FILE in *.fastq; do
    NAME=$(basename "$FILE")
    COUNT=$(( $(wc -l < "$FILE") / 4 ))
    echo "$NAME,$COUNT,trimmed" >> ../reads.csv
done


# JOB-ID: 51535798
# script file: /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024
#trimmed read seqs in folder: /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed
# (raw seqs also in scratch workspace for now & lab hard drive. will transition to zipped in project)

### Multiqc 
- https://github.com/MultiQC/MultiQC
- summarizes fastqc reports to one html file
- doesn't perform analysis just summarizes already existing reports

In [ ]:
# could probably add this to fastqc bash script next time 

In [ ]:
# run multiqc in dir with fastqc results
module load conda/latest
conda activate multiqc 

cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed
multiqc . 

### Read depth

In [ ]:
# calculate read depth?

In [1]:
import os
import pandas as pd

In [38]:
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw

/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw


In [39]:
ls raw | head -n 1 > file

ls: write error: Broken pipe


In [40]:
ls

file  raw/  reads  sampleids.txt  trimmed/


In [42]:
!head file

052022_BEL_CBC_T1_10_PSTR_R1_001.fastq


In [ ]:
cat file | sed 's/_S[0-9]\+_R[12]//'

In [47]:
cat raw/052022_BEL_CBC_T1_10_PSTR_R1_001.fastq | echo $(wc -l)/4|bc

68572842


In [27]:
!for FILE in *; do echo $(wc -l)/4|bc; done > /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/reads


^C


In [ ]:
!ls -1

In [ ]:
# on zipped files
!for FILE in *; do echo $(zcat $FILE|wc -l)/4|bc; done > reads

In [21]:
# !ls *.fastq > samples
# my sampleid: sampleids.txt

In [25]:
!paste samples reads > sample_reads

In [ ]:
!ls | head -n -1

In [29]:
!head -n -1 sample_reads > sample_reads

In [ ]:
for FILE in *; do echo $(zcat $FILE|wc -l)/4|bc; done > reads
ls *.gz > samples
paste samples reads > sample_reads

In [35]:
cd //project/pi_sarah_gignouxwolfsohn_uml_edu/Raw_sequences/SCTLD_raw/renamed/pilot_copy/

11272023/  mcav/  pilot_copy/


In [ ]:
for FILE in *; do echo $(cat $FILE|wc -l)/4|bc; done > reads
ls *.gz > samples
paste samples reads > sample_reads

In [9]:
# investigate averages 
os.chdir('//project/pi_sarah_gignouxwolfsohn_uml_edu/Raw_sequences/SCTLD_raw/renamed/pilot_copy')

In [17]:
# in bash 
!awk '{sum += $1} END {print sum/NR}' sample_reads
print("pilot")

1.70618e+07
pilot


In [27]:
# investigate averages 
os.chdir('//project/pi_sarah_gignouxwolfsohn_uml_edu/Raw_sequences/SCTLD_raw/renamed/mcav')

In [28]:
!awk '{sum += $1} END {print sum/NR}' reads
print("mcav")

2.30272e+07
mcav


In [29]:
# investigate averages 
os.chdir('//project/pi_sarah_gignouxwolfsohn_uml_edu/Raw_sequences/SCTLD_raw/renamed/11272023')
!awk '{sum += $1} END {print sum/NR}' reads
print("11272023")

2.58825e+07
11272023
